# Example Barnes Double Gamma Surface

Canonical Barnes and double-gamma notebook for the production special-function tranche.

## Scope

This notebook is the canonical example surface for `example_barnes_double_gamma_surface`. It runs against the repo source tree through `/src`, shows direct public API usage, summarizes validation and benchmark status, and includes visual summaries.

In [ ]:
import io
import json
import os
import re
import subprocess
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'arbplusjax').exists():
            return p
    raise RuntimeError(f'Could not locate repo root from: {start}')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
os.chdir(REPO_ROOT)

PYTHON = os.getenv('ARBPLUSJAX_PYTHON', sys.executable)
JAX_MODE = os.getenv('JAX_MODE', 'cpu').strip().lower()
JAX_DTYPE = os.getenv('JAX_DTYPE', 'float64').strip().lower()
RUN_ENV = os.environ.copy()
RUN_ENV['PYTHONPATH'] = str(REPO_ROOT / 'src') + os.pathsep + RUN_ENV.get('PYTHONPATH', '')
if JAX_MODE == 'cpu':
    RUN_ENV['JAX_PLATFORMS'] = 'cpu'
elif JAX_MODE == 'gpu':
    RUN_ENV['JAX_PLATFORMS'] = 'cuda'
RUN_ENV['JAX_ENABLE_X64'] = '1' if JAX_DTYPE == 'float64' else '0'
EXAMPLE_INPUT_ROOT = REPO_ROOT / 'examples' / 'inputs' / 'example_barnes_double_gamma_surface'
EXAMPLE_OUTPUT_ROOT = REPO_ROOT / 'examples' / 'outputs' / 'example_barnes_double_gamma_surface'
EXAMPLE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd: list[str], *, capture: bool = False):
    print('[cmd]', ' '.join(cmd))
    return subprocess.run(cmd, cwd=REPO_ROOT, env=RUN_ENV, text=True, capture_output=capture, check=True)


## Environment

The notebook reports interpreter, selected JAX mode, and the active backend/device view. Canonical retained execution in this repo state is CPU-oriented, but the notebook calling pattern remains CPU/GPU portable and explicitly parameterized for `float32` and `float64`.

In [ ]:
SUPPORTED_JAX_MODES = ('cpu', 'gpu')
SUPPORTED_JAX_DTYPES = ('float32', 'float64')
if JAX_MODE not in SUPPORTED_JAX_MODES:
    raise ValueError(f'Unsupported JAX_MODE: {JAX_MODE}')
if JAX_DTYPE not in SUPPORTED_JAX_DTYPES:
    raise ValueError(f'Unsupported JAX_DTYPE: {JAX_DTYPE}')
print('python:', PYTHON)
print('jax_mode:', JAX_MODE)
print('jax_dtype:', JAX_DTYPE)
print('supported_jax_modes:', SUPPORTED_JAX_MODES)
print('supported_jax_dtypes:', SUPPORTED_JAX_DTYPES)
print('validation_slice:', 'cpu_current__gpu_portable_contract')
runtime = run([PYTHON, 'tools/check_jax_runtime.py'], capture=True)
print(runtime.stdout)
runtime_payload = json.loads(runtime.stdout)
(EXAMPLE_OUTPUT_ROOT / f'runtime_{JAX_MODE}.json').write_text(json.dumps(runtime_payload, indent=2) + '\n', encoding='utf-8')

## Direct Usage

Exercise Barnes G and double-gamma public surfaces, including diagnostics.

In [ ]:
import jax.numpy as jnp
from arbplusjax import api, double_gamma

z = jnp.asarray(1.7 + 0.1j, dtype=jnp.complex128)
tau = jnp.float64(0.5)
barnes_results = {
    'barnesg': api.eval_point('acb_barnes_g', z),
    'double_gamma_legacy': double_gamma.bdg_barnesdoublegamma(z, tau, prec_bits=80),
    'double_gamma_ifj': double_gamma.ifj_barnesdoublegamma(z, tau, dps=60),
}
diagnostics = double_gamma.ifj_barnesdoublegamma_diagnostics(0.2 + 0.05j, 1.0, dps=60, max_m_cap=256)
display(barnes_results)
display({'m_base': diagnostics.m_base, 'm_used': diagnostics.m_used, 'n_shift': diagnostics.n_shift, 'm_capped': diagnostics.m_capped})

## Production Pattern

Barnes and double-gamma usage should keep the chosen implementation, precision, and diagnostics policy explicit. These are not hot scalar helpers; production code should avoid switching precision knobs per call unless that is a deliberate fallback path.

In [ ]:
barnes_service = {
    'legacy_fixed_prec': double_gamma.bdg_barnesdoublegamma(z, tau, prec_bits=80),
    'ifj_fixed_dps': double_gamma.ifj_barnesdoublegamma(z, tau, dps=60),
}
display(barnes_service)

## Extending Benchmarks

To extend Barnes/double-gamma benchmarks, add new printed metrics in `benchmark_barnes_double_gamma.py` or split out a schema-backed service benchmark if repeated-call API usage becomes a primary concern.

## Fast JAX Point Pattern

Barnes-family point evaluation is still more specialized than the lightweight scalar helpers. The fast-JAX proof here is a family-owned compiled derivative path rather than a generic repeated batch service.

In [ ]:
import jax
barnes_fast = jax.jit(lambda xs: jax.vmap(lambda t: jnp.real(double_gamma.ifj_barnesdoublegamma(jnp.asarray(t + 0.05j, dtype=jnp.complex128), 1.0, dps=60)))(xs))
barnes_x = jnp.linspace(0.8, 2.0, 8, dtype=jnp.float64)
barnes_fast_out = barnes_fast(barnes_x)
display({'jit_shape': barnes_fast_out.shape, 'finite': bool(jnp.all(jnp.isfinite(barnes_fast_out)))})

## AD Product Pattern

Barnes-family AD should be shown explicitly because these surfaces are more specialized and their derivative expectations are less obvious to users. This section differentiates the IFJ log-form path over a real sweep and plots primal versus gradient.

In [ ]:
import jax
def barnes_loss(xv):
    val = double_gamma.ifj_barnesdoublegamma(jnp.asarray(xv + 0.05j, dtype=jnp.complex128), 1.0, dps=60)
    return jnp.real(val)
x_sweep = jnp.linspace(0.8, 2.0, 24, dtype=jnp.float64)
primal_vals = jax.vmap(barnes_loss)(x_sweep)
grad_vals = jax.vmap(jax.grad(barnes_loss))(x_sweep)
ad_df = pd.DataFrame({'x': np.asarray(x_sweep), 'primal': np.asarray(primal_vals), 'grad': np.asarray(grad_vals)})
display(ad_df.head())
ax = ad_df.plot(x='x', y=['primal', 'grad'], figsize=(8, 4), title='Barnes AD Validation')
plt.tight_layout()
plt.savefig(EXAMPLE_OUTPUT_ROOT / f'ad_validation_{JAX_MODE}.png', dpi=160, bbox_inches='tight')
plt.show()

## Validation Summary

Run the Barnes/double-gamma contract tests.

In [ ]:
tests = run([
    PYTHON, '-m', 'pytest', '-q',
    'tests/test_barnes_tier1.py',
    'tests/test_double_gamma_contracts.py',
    'tests/test_shahen_double_gamma.py',
], capture=True)
print(tests.stdout)
if tests.stderr:
    print(tests.stderr)
(EXAMPLE_OUTPUT_ROOT / f'pytest_{JAX_MODE}.txt').write_text(tests.stdout + ('\n' + tests.stderr if tests.stderr else ''), encoding='utf-8')

## Benchmark Summary

Run the Barnes/double-gamma benchmark and parse the key-value output.

In [ ]:
completed = run([PYTHON, 'benchmarks/benchmark_barnes_double_gamma.py'], capture=True)
print(completed.stdout)
rows = []
for line in completed.stdout.splitlines():
    if ': ' not in line:
        continue
    key, value = line.split(': ', 1)
    try:
        rows.append({'metric': key, 'value': float(value)})
    except ValueError:
        rows.append({'metric': key, 'value': value})
bench_df = pd.DataFrame(rows)
bench_df.to_csv(EXAMPLE_OUTPUT_ROOT / f'barnes_double_gamma_summary_{JAX_MODE}.csv', index=False)
display(bench_df)

## Optional Diagnostics

The IFJ diagnostics object is the primary hardening signal for this notebook.

In [ ]:
summary_lines = [
    f'# Example Barnes Double Gamma Surface Summary ({JAX_MODE})',
    '',
    f'- backend: `{runtime_payload["platform"]}`',
    f'- benchmark_rows: `{len(bench_df)}`',
    f'- diagnostics_m_used: `{diagnostics.m_used}`',
    '',
    '## Key Metrics',
    '',
]
for row in bench_df.to_dict(orient='records')[:12]:
    summary_lines.append(f"- `{row['metric']}`: `{row['value']}`")
(EXAMPLE_OUTPUT_ROOT / f'summary_{JAX_MODE}.md').write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')
display('\n'.join(summary_lines[:14]))